In [5]:
import numpy as np
import pandas as pd

# =========================
# Load data
# =========================

X = np.load(
    r"C:\Users\USER\Desktop\資訊所實習\計畫\資料探勘\Ravindra2021.raw_count.stdprep.h5ad的細胞標籤\將mock拿掉重跑\GCN_features.npy"
)

metadata = pd.read_csv(
    r"C:\Users\USER\Desktop\資訊所實習\計畫\資料探勘\Ravindra2021.raw_count.stdprep.h5ad的細胞標籤\將mock拿掉重跑\GCN_metadata.csv",
    index_col=0
)

print("X shape:", X.shape)
print("Metadata shape:", metadata.shape)

print(metadata["label"].value_counts())

X shape: (51981, 3000)
Metadata shape: (51981, 5)
label
-1    42769
 0     7505
 1     1707
Name: count, dtype: int64


In [6]:
label_mask = metadata["label"].isin([0, 1])

X_labeled = X[label_mask.values]

y_labeled = metadata.loc[
    label_mask,
    "label"
].values

cell_names = metadata.index[label_mask]

In [7]:
print("X_labeled:", X_labeled.shape)
print("y_labeled:", y_labeled.shape)
print("Label counts:")
print(pd.Series(y_labeled).value_counts())

X_labeled: (9212, 3000)
y_labeled: (9212,)
Label counts:
0    7505
1    1707
Name: count, dtype: int64


In [8]:
from sklearn.model_selection import train_test_split

indices = np.arange(len(y_labeled))

train_idx, test_idx = train_test_split(
    indices,
    test_size=0.2,
    stratify=y_labeled,
    random_state=42
)

X_train = X_labeled[train_idx]
X_test = X_labeled[test_idx]

y_train = y_labeled[train_idx]
y_test = y_labeled[test_idx]

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (7369, 3000)
Testing: (1843, 3000)


In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score
)

model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model.fit(
    X_train,
    y_train
)

# 預測
y_pred = model.predict(X_test)

# infected probability
y_prob = model.predict_proba(X_test)[:, 1]

In [10]:
print("Confusion Matrix:")
print(
    confusion_matrix(
        y_test,
        y_pred
    )
)

print()

print("Classification Report:")
print(
    classification_report(
        y_test,
        y_pred
    )
)

roc_auc = roc_auc_score(
    y_test,
    y_prob
)

pr_auc = average_precision_score(
    y_test,
    y_prob
)

print("ROC-AUC:", round(roc_auc, 4))
print("PR-AUC:", round(pr_auc, 4))

Confusion Matrix:
[[1501    0]
 [   2  340]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1501
           1       1.00      0.99      1.00       342

    accuracy                           1.00      1843
   macro avg       1.00      1.00      1.00      1843
weighted avg       1.00      1.00      1.00      1843

ROC-AUC: 0.9998
PR-AUC: 0.9992


In [14]:
# ==========================================
# Predict unknown cells
# ==========================================

# 找出 unknown cells
unknown_mask = metadata["label"] == -1

# unknown 的 gene expression
X_unknown = X[unknown_mask.values]

# unknown 的 metadata
unknown_metadata = metadata.loc[
    unknown_mask
].copy()

print("Unknown cells:", X_unknown.shape[0])


# ==========================================
# Logistic prediction
# ==========================================

# 預測 label
unknown_pred = model.predict(X_unknown)

# 預測感染機率
unknown_prob = model.predict_proba(
    X_unknown
)[:, 1]


# ==========================================
# 把結果放回 metadata
# ==========================================

unknown_metadata["predict_label"] = unknown_pred

unknown_metadata["infect_prob"] = unknown_prob


# ==========================================
# 分類結果統計
# ==========================================

print("")
print("Unknown prediction:")
print(
    unknown_metadata["predict_label"]
    .value_counts()
)

print("")
print("Unknown infection probability:")
print(
    unknown_metadata["infect_prob"].describe()
)


# ==========================================
# 匯出
# ==========================================

save_path = (
    r"C:\Users\USER\Desktop\資訊所實習"
    r"\計畫\資料探勘"
    r"\Ravindra2021.raw_count.stdprep.h5ad的細胞標籤"
    r"\將mock拿掉重跑"
    r"\其他機器學習模型"
    r"\Logistic_unknown_prediction.csv"
)

unknown_metadata.to_csv(
    save_path,
    encoding="utf-8-sig"
)

print("")
print("CSV 已輸出:")
print(save_path)
print("資料大小:", unknown_metadata.shape)

Unknown cells: 42769

Unknown prediction:
predict_label
1    24283
0    18486
Name: count, dtype: int64

Unknown infection probability:
count    4.276900e+04
mean     5.619528e-01
std      4.342051e-01
min      1.277084e-10
25%      2.992934e-02
50%      7.497771e-01
75%      9.939830e-01
max      1.000000e+00
Name: infect_prob, dtype: float64

CSV 已輸出:
C:\Users\USER\Desktop\資訊所實習\計畫\資料探勘\Ravindra2021.raw_count.stdprep.h5ad的細胞標籤\將mock拿掉重跑\其他機器學習模型\Logistic_unknown_prediction.csv
資料大小: (42769, 7)
